# Diagnosis questions: what can the data say about *why*?
**Internet of Production – AI, week 1 (worked example)** · accompanies the slide “Our questions: what can the data answer?”

In the group activity the questions of the head of pre-production were sorted into *describe – diagnose – predict – recommend – act* and then into *answerable with the data set* and *not answerable*. This notebook answers the six **green questions of the column “Diagnose”**:

| # | Question | Data |
|---|---|---|
| 1 | Which parts cause which defects? | `Printproduction_QM_scrap.csv` |
| 2 | Which printing defect is linked to which printer (e.g. warping)? | QM file |
| 3 | Is there a correlation between extruder blockage and connection interrupted? | `Printer_*_error.csv` |
| 4 | How many parts failed, and why (defect type)? | orders, QM file, error files |
| 5 | What are the differences between the printers? | minute data, error files, orders, QM file |
| 6 | Which part on which printer, to minimise failures? | orders, QM file |

**Rule of this notebook:** every answer states the number, its **uncertainty** and what the data **cannot** tell. Diagnosis asks for causes; a data set of observations shows associations. Where the data do not support a statement we say so.

## Setup and data check

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

DATA = Path('../../00_Data/printfarm')
DEF = ['material breakout', 'support structure remains', 'under-extrusion', 'warpage']
rng = np.random.default_rng(0)

orders = pd.read_csv(DATA / 'Printproduction.csv', sep=';', decimal=',').dropna(how='all')
orders['Start'] = pd.to_datetime(orders['Start'], format='%d.%m.%Y %H:%M')
orders['Printer'] = orders['Printer'].astype(int)
orders['failed'] = (orders['Status'] == 'failed').astype(int)
qm = pd.read_csv(DATA / 'Printproduction_QM_scrap.csv', sep=';')

errors = []
for k in (0, 1, 2):
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    e['Printer'] = k
    errors.append(e)
errors = pd.concat(errors)
part_order = sorted(orders['BOM Item'].unique(), key=lambda s: int(s[:-1]))   # 1a, 2a, ..., 10a

print('orders:', len(orders), ' failed:', orders['failed'].sum(), ' planned parts:', int(orders['Lot size'].sum()))
print('orders with a QM record:', qm['Order'].isin(orders['Order']).sum(), ' of', (orders['failed'] == 0).sum(), 'completed orders')
print('QM record for a failed order:', qm['Order'].isin(orders.loc[orders['failed'] == 1, 'Order']).sum())
print('lot size = OK + not OK:', bool((qm['Lot size'] == qm['OK'] + qm['not OK']).all()))
print('not OK = sum of the four defect columns:', bool((qm['not OK'] == qm[DEF].sum(axis=1)).all()))

**What do these checks mean for the analysis?**

- Quality control exists only for **completed** orders: 1,001 of 1,028 completed orders have a QM record, **no failed order has one**. Defects therefore cannot explain failed orders; the two kinds of loss have different causes and must be counted separately.
- 27 completed orders (120 parts) have **no QM record**: their quality is unknown, not “OK”.
- Lot size = OK + not OK, so the QM columns count **parts**, not orders. Not OK = sum of the four defect columns, so every scrap part has **exactly one** defect type; the defect types are mutually exclusive categories.

## Two kinds of loss, and why uncertainty needs care
- **Failed order:** the job was aborted, all parts of the lot are lost; there is **no QM record**. The reason is a disturbance of the printer (see question 4).
- **Scrap part:** the order was completed, but a part is *not OK* in quality control; each such part has exactly **one** defect type.

Parts of one order are **not independent** (same print job, same lot of 3, 6 or 12 parts). A classical test that counts every part as an independent observation is therefore too optimistic. We use two simple tools that respect this:
1. **Order bootstrap:** to get an interval for a rate, we resample *orders* with replacement and recompute the rate.
2. **Permutation test:** to test whether a factor (e.g. the printer) matters, we shuffle it among orders that are alike in the other factor (e.g. the same part) and see how large the differences become by chance.

In [ ]:
def wilson(k, n, z=1.96):
    """Wilson interval for a share k/n of independent cases (here: orders)."""
    if n == 0: return (np.nan, np.nan)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

def order_bootstrap(df, group, cols, B=2000, level=95):
    """Rate = sum(cols) / sum(parts) per group, interval by resampling ORDERS within the group."""
    out = []
    for name, g in df.groupby(group):
        parts = g['Lot size'].to_numpy(float)
        X = g[cols].to_numpy(float)
        idx = rng.integers(0, len(g), size=(B, len(g)))
        r = X[idx].sum(axis=1) / parts[idx].sum(axis=1)[:, None]
        lo, hi = np.percentile(r, [(100 - level) / 2, 100 - (100 - level) / 2], axis=0)
        out.append(pd.DataFrame({group: name, 'defect': cols, 'rate': X.sum(axis=0) / parts.sum(), 'lo': lo, 'hi': hi, 'orders': len(g), 'parts': int(parts.sum())}))
    return pd.concat(out, ignore_index=True)

def permutation_p(stat, df, factor, strata, n=1000):
    """p-value: how often a shuffle of `factor` within `strata` gives a statistic at least as large as observed."""
    s0 = stat(df)
    sims = np.empty(n)
    for i in range(n):
        d = df.copy()
        d[factor] = d.groupby(strata)[factor].transform(lambda x: rng.permutation(x.to_numpy()))
        sims[i] = stat(d)
    return s0, (np.sum(sims >= s0) + 1) / (n + 1)

## Question 1: Which parts cause which defects?
Defect rate = defective parts of this type / all parts of this part number (parts, not orders). The 99 % interval from the order bootstrap marks a cell as **reliably above average** when its whole interval lies above the overall rate (99 % because we look at 40 cells at once).

In [ ]:
cols = DEF + ['not OK']
overall = qm[cols].sum() / qm['Lot size'].sum()
ci = order_bootstrap(qm, 'BOM Item', cols, level=99)
rate = ci.pivot(index='BOM Item', columns='defect', values='rate').loc[part_order, cols] * 100
lo = ci.pivot(index='BOM Item', columns='defect', values='lo').loc[part_order, cols] * 100
reliable = lo.gt(overall[cols] * 100, axis=1)          # whole interval above the overall rate

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(rate.values, cmap='Oranges', aspect='auto')
ax.set_xticks(range(len(cols))); ax.set_xticklabels(['breakout', 'support remains', 'under-extrusion', 'warpage', 'all scrap'], rotation=20)
ax.set_yticks(range(len(rate))); ax.set_yticklabels([f"{p} (n = {int(qm.loc[qm['BOM Item'] == p, 'Lot size'].sum())})" for p in rate.index])
for i in range(rate.shape[0]):
    for j in range(rate.shape[1]):
        ax.text(j, i, f"{rate.iat[i, j]:.1f}" + (' *' if reliable.iat[i, j] else ''), ha='center', va='center', fontsize=9, fontweight='bold' if reliable.iat[i, j] else 'normal')
ax.set_title('Defective parts per 100 parts; * = reliably above the overall rate')
plt.colorbar(im, label='% of parts'); plt.tight_layout(); plt.show()

print('overall rate (% of parts):', (overall * 100).round(2).to_dict())
print('reliably above average:', [(p, c) for p in rate.index for c in cols if reliable.loc[p, c]])

**Which part-defect combinations are reliable, and which are only visible in the table?**

**Reliable:** only **part 1a has more warpage** than average (7.5 % of its parts against 3.8 % overall; 1a is always printed as a lot of 1, 414 parts, so this cell is not distorted by clusters). **Visible but not reliable:** support structure remains is highest for 8a (9.0 %), 5a (8.6 %), 6a (8.5 %) and 9a (7.5 %) against 5.3 % overall; under-extrusion for 3a (4.6 %); breakout for 5a, 8a and 2a (2.4 to 2.7 % against 1.3 %). Their intervals contain the overall rate, because a few bad orders of 6 or 12 parts dominate a cell. Part 5a shows no warpage at all (0.0 %), which is as striking as it is unreliable.
**Limit:** the data show *which* part shows *which* defect more often, not *why* (geometry, orientation and print settings are not recorded). With 40 cells, one or two “findings” at the 99 % level would appear by chance alone; only 1a/warpage is a candidate for a root-cause analysis (for example orientation or bed adhesion of this part).

## Question 2: Which printing defect is linked to which printer?
Two traps: (a) the printers print **different part mixes**, so a printer can look bad only because it printed difficult parts; (b) the parts of one order are not independent. We therefore compare each printer with its **expected** number of defects given the part mix it printed (observed / expected), and test with a permutation test that shuffles the printer among orders of the same part.

In [ ]:
part_rate = qm.groupby('BOM Item')[DEF].sum().div(qm.groupby('BOM Item')['Lot size'].sum(), axis=0)   # defect rate per part number

def expected_by_part_mix(df):
    parts = df.groupby(['Printer', 'BOM Item'])['Lot size'].sum().unstack(fill_value=0).reindex(columns=part_rate.index, fill_value=0)
    return parts.dot(part_rate)

def printer_defect_stat(df):
    o = df.groupby('Printer')[DEF].sum()
    e = expected_by_part_mix(df)
    return (((o - e) ** 2) / e).to_numpy().sum()

obs = qm.groupby('Printer')[DEF].sum()
exp = expected_by_part_mix(qm)
print('observed defects per printer'); display(obs)
print('observed / expected given the part mix'); display((obs / exp).round(2))

ci_p = order_bootstrap(qm, 'Printer', DEF + ['not OK'], level=95)
fig, ax = plt.subplots(figsize=(9, 3.6))
for j, c in enumerate(DEF + ['not OK']):
    for k in (0, 1, 2):
        r = ci_p[(ci_p['Printer'] == k) & (ci_p['defect'] == c)].iloc[0]
        ax.errorbar(j + (k - 1) * 0.22, r['rate'] * 100, yerr=[[r['rate'] * 100 - r['lo'] * 100], [r['hi'] * 100 - r['rate'] * 100]], fmt='o', color=['#003A79', '#009EE0', '#EE7F00'][k], capsize=3, label=f'printer {k}' if j == 0 else None)
ax.set_xticks(range(5)); ax.set_xticklabels(['breakout', 'support remains', 'under-extrusion', 'warpage', 'all scrap'])
ax.set_ylabel('% of parts'); ax.set_title('Defect rate per printer with 95 % interval (order bootstrap)'); ax.legend(); plt.tight_layout(); plt.show()

s0, p = permutation_p(printer_defect_stat, qm, 'Printer', 'BOM Item')
print(f'overall test (printer vs defect profile, given the part): statistic {s0:.1f}, permutation p = {p:.2f}')

**Is there a printer with a reliably different defect profile? What does a naive chi-square test say, and why is it misleading?**

**No printer has a reliably different defect profile.** Observed / expected given the part mix: printer 0 has more warpage (1.42) and under-extrusion (1.45) but fewer support remains (0.63); printer 1 has little under-extrusion (0.44); printer 2 has little warpage (0.56) and more support remains (1.28). The permutation test over all defect types gives p = 0.17, and every printer’s interval in the plot contains the average. The tempting reading “printer 0 warps, printer 2 does not” is within order-to-order chance.
The **naive chi-square test gives p = 6e-10** and would declare a strong printer effect. It is misleading for two reasons: it treats the 1,468 to 1,723 parts per printer as independent although they come from only 317 to 342 orders (lots of 1, 3, 6 or 12), and it ignores that the printers print different part mixes. A tiny p-value from a wrong model is not evidence.
**Limit:** a real printer effect of moderate size could still be hidden; the data only say that they do not show one.

In [ ]:
naive = stats.chi2_contingency(qm.groupby('Printer')[DEF + ['OK']].sum().to_numpy())[1]
print(f'naive chi-square (every part independent, part mix ignored): p = {naive:.1e}')

## Question 3: Is there a correlation between extruder blockage and connection interrupted?
Both disturbances are time stamps in the error files. If one caused or followed the other, interruptions should lie **closer to blockages than chance** would put them. Test: for each blockage take the distance to the nearest connection interruption; compare the share within ±24 h with what happens if the interruptions were placed **at random** in the same period (1000 draws, same number of interruptions).

In [ ]:
def nearest_hours(b, c):
    """distance in hours from every time in b to the nearest time in c (both sorted datetime64 arrays)"""
    i = np.searchsorted(c, b)
    left, right = c[np.clip(i - 1, 0, len(c) - 1)], c[np.clip(i, 0, len(c) - 1)]
    return np.minimum(np.abs(b - left), np.abs(b - right)) / np.timedelta64(1, 'h')

t0, t1 = pd.Timestamp('2024-01-05'), pd.Timestamp('2025-05-14')
window = 24
rows = []
fig, axes = plt.subplots(3, 1, figsize=(10, 4), sharex=True)
for k, ax in zip((0, 1, 2), axes):
    e = errors[errors['Printer'] == k]
    blk = np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].to_numpy())
    con = np.sort(e.loc[e['Error'] == 'Connection interrupted', 'Timestamp'].to_numpy())
    ax.vlines(blk, 0.1, 0.9, color='#E2001A', label='extruder blocked'); ax.vlines(con, -0.9, -0.1, color='#009EE0', label='connection interrupted')
    ax.set_yticks([]); ax.set_ylabel(f'printer {k}')
    if k == 0: ax.legend(loc='upper right', ncol=2, fontsize=8)
    if len(blk) < 5 or len(con) < 1:
        rows.append({'printer': k, 'blockages': len(blk), 'interruptions': len(con), 'share within 24 h': np.nan, 'expected by chance': np.nan, 'p': np.nan}); continue
    share = np.mean(nearest_hours(blk, con) <= window)
    sims = []
    for _ in range(1000):
        r = np.sort(rng.integers(t0.value, t1.value, size=len(con)).astype('datetime64[ns]'))
        sims.append(np.mean(nearest_hours(blk, r) <= window))
    rows.append({'printer': k, 'blockages': len(blk), 'interruptions': len(con), 'share within 24 h': round(share, 3), 'expected by chance': round(np.mean(sims), 3), 'p': round((np.sum(np.array(sims) >= share) + 1) / 1001, 3)})
plt.tight_layout(); plt.show()
pd.DataFrame(rows).set_index('printer')

**Is there a connection? What would be needed to show a causal link, and what would be a possible common cause?**

**No evidence of a connection.** Printer 1 (34 blockages, 11 interruptions): 0 of 34 blockages have an interruption within 24 h, chance gives about 4 % (p = 1.0, no excess). Printer 2 (18 blockages, 24 interruptions): 3 of 18 (17 %) against 9 % expected by chance, p = 0.20, which is within chance. Printer 0 has only 2 blockages and cannot be tested.
The picture is rather one of **different disturbance patterns per printer**: printer 0 shows almost only interruptions (29 of 35 events), printer 1 almost only blockages (34 of 50). That is compatible with printer-specific causes (network or cable for interruptions, hot end or filament path for blockages), not with one disturbance triggering the other.
**Limit:** correlation in time is not causation, and absence of evidence is not evidence of absence: with 11 and 18 events per printer only a strong coupling would be visible. A causal statement would need the logs of the network and the hot end. Also, blockages occur only while printing, whereas interruptions can occur in any state; the random model above ignores this.

## Question 4: How many parts failed, and why (defect type)?
All planned parts end in one of four states. For failed orders the reason is the disturbance of the printer during the order (as in exercise 3: time window of the order ± 5 minutes); for scrap parts it is the defect type of the QM file.

In [ ]:
orders['End'] = orders['Start'] + pd.to_timedelta(orders['Actual Duration'], unit='m')
tol = pd.Timedelta(minutes=5)

def disturbance(o):
    e = errors[(errors['Printer'] == o['Printer']) & (errors['Timestamp'] >= o['Start'] - tol) & (errors['Timestamp'] <= o['End'] + tol)]
    return e['Error'].iloc[0] if len(e) else 'none'

orders['disturbance'] = orders.apply(disturbance, axis=1)
failed = orders[orders['failed'] == 1]
by_cause = failed.groupby('disturbance').agg(orders=('Order', 'size'), parts_lost=('Lot size', 'sum')).sort_values('parts_lost', ascending=False)
print('failed orders and parts lost, by disturbance during the order'); display(by_cause)
print('completed orders with a disturbance during the order:', int(((orders['failed'] == 0) & (orders['disturbance'] != 'none')).sum()))

no_qm = orders[(orders['failed'] == 0) & (~orders['Order'].isin(qm['Order']))]
fate = pd.Series({
    'OK': qm['OK'].sum(),
    'scrap: support structure remains': qm['support structure remains'].sum(),
    'scrap: warpage': qm['warpage'].sum(),
    'scrap: under-extrusion': qm['under-extrusion'].sum(),
    'scrap: material breakout': qm['material breakout'].sum(),
    'lost: failed order': failed['Lot size'].sum(),
    'unknown: completed, no QM record': no_qm['Lot size'].sum()})
planned = int(orders['Lot size'].sum())
print('planned parts:', planned, ' sum of the states:', int(fate.sum()))
fig, ax = plt.subplots(figsize=(9, 3.6))
colors = ['#7AB51D', '#EE7F00', '#EE7F00', '#EE7F00', '#EE7F00', '#E2001A', '#9AA0A6']
ax.barh(fate.index[::-1], fate.values[::-1], color=colors[::-1])
for i, v in enumerate(fate.values[::-1]): ax.text(v + 30, i, f'{int(v)}  ({v / planned * 100:.1f} %)', va='center', fontsize=9)
ax.set_xlim(0, fate.max() * 1.25); ax.set_xlabel('parts'); ax.set_title('What happens to the planned parts'); plt.tight_layout(); plt.show()

**How many parts are lost in total, and which kind of loss is larger? What does the missing QM record of 27 completed orders mean?**

Of **5,477 planned parts** 4,181 (76.3 %) are OK. **1,296 (23.7 %) are not**:
- **572 (10.4 %) lost in 134 failed orders**, by disturbance during the order: connection interrupted 64 orders / 295 parts, extruder blocked 54 / 212, empty filament 15 / 53, crash 1 / 12. In **no** completed order a disturbance occurred, so a disturbance during the order is what separates failed from completed orders.
- **604 (11.0 %) scrap** in completed orders: support structure remains 253 (4.6 %), warpage 183 (3.3 %), under-extrusion 104 (1.9 %), breakout 64 (1.2 %).
- **120 (2.2 %) unknown**: 27 completed orders without a QM record.
The two losses are about equally large. Connection interruptions cost more parts (295) than extruder blockages (212), although only blockages show a warning sign in the energy data; an early warning for blockages could therefore address at most 212 of 572 lost parts (37 %), before any false alarms. **Limit:** the cause of a failed order is the disturbance *during* the order (time window of the error file); why the disturbance happened is not in the data.

## Question 5: What are the differences between the printers?
Five views: workload and states (minute data), energy while printing, disturbances (number and type), failed orders, scrap. For each difference we ask: is it larger than chance?

In [ ]:
rows, state_share = [], {}
for k in (0, 1, 2):
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', usecols=['State', 'Total_Power']).dropna(how='all')
    d['E'] = d['Total_Power'].diff() * 1000                    # Wh per minute (unit assumed kWh)
    state_share[k] = d['State'].value_counts(normalize=True).mul(100)
    rows.append({'printer': k, 'printing hours': round((d['State'] == 'printing').sum() / 60), 'energy while printing (Wh/min)': round(d.loc[d['State'] == 'printing', 'E'].mean(), 3)})
minute_view = pd.DataFrame(rows).set_index('printer')
print('states, share of time in %'); display(pd.DataFrame(state_share).round(1).loc[['printing', 'ready', 'error', 'setup', 'idle']])
display(minute_view)

types = errors.groupby(['Error', 'Printer']).size().unstack(fill_value=0)
print('disturbances (number of events)'); display(types)
print(f"chi-square type x printer (events are separate time stamps): p = {stats.chi2_contingency(types.to_numpy())[1]:.1e}")

In [ ]:
fail_rate = orders.groupby('BOM Item')['failed'].mean()

def printer_failure_stat(df):
    expected = df.groupby('Printer')['BOM Item'].apply(lambda s: fail_rate.reindex(s).sum())
    observed = df.groupby('Printer')['failed'].sum()
    return (((observed - expected) ** 2) / expected).sum()

summary = pd.DataFrame({
    'orders': orders.groupby('Printer').size(),
    'failed orders (%)': (orders.groupby('Printer')['failed'].mean() * 100).round(1),
    'scrap parts (%)': (qm.groupby('Printer')['not OK'].sum() / qm.groupby('Printer')['Lot size'].sum() * 100).round(1),
    'parts lost in failed orders (% of planned)': (orders[orders['failed'] == 1].groupby('Printer')['Lot size'].sum() / orders.groupby('Printer')['Lot size'].sum() * 100).round(1)})
display(summary)
s0, p_fail = permutation_p(printer_failure_stat, orders, 'Printer', 'BOM Item')
print(f'failed orders: printer effect given the part: permutation p = {p_fail:.2f}')

**Which differences between the printers are real, and which are within chance?**

**Real differences (far beyond chance): the disturbance pattern.** Printer 0 has almost only connection interruptions (29 of 35 events), printer 1 almost only extruder blockages (34 of 50), printer 2 a mixture (24 interruptions, 18 blockages); chi-square p = 6e-7. Because blockages stop a printer for long, printer 1 spends **7.2 % of its time in the error state** against 4.0 % (printer 0) and 5.2 % (printer 2), and it loses most printing capacity.
**No relevant difference:** energy while printing (1.641, 1.645, 1.643 Wh/min, with a minute-to-minute spread of 0.14), share of time printing (32.4, 32.7, 34.3 %), scrap rate (13.2, 12.2, 12.5 %).
**Within chance:** failed orders (9.1, 12.5, 13.0 %); given the part mix the permutation test gives p = 0.20. Printer 0 looks best, but 385 orders cannot separate 9 % from 13 %.
**Limit:** the printers are not equal experiments: part mix, period and operators may differ. “Printer 1 is the problem printer” is true for blockages and time lost in error, not for scrap and not (reliably) for failed orders.

## Question 6: Which part on which printer, to minimise failures?
A recommendation “print part X on printer Y” needs a difference between printers **for the same part** that is larger than chance. Table: failure rate of the orders per part and printer with the number of orders; Wilson 95 % interval for the best and the worst printer of each part. If the intervals overlap, the data do not justify a preference.

In [ ]:
tab = orders.groupby(['BOM Item', 'Printer'])['failed'].agg(['sum', 'size']).unstack('Printer').reindex(part_order)
rows = []
for p in part_order:
    k = tab.loc[p, ('sum', slice(None))].to_numpy(); n = tab.loc[p, ('size', slice(None))].to_numpy()
    r = k / n
    best, worst = int(np.argmin(r)), int(np.argmax(r))
    lo_b, hi_b = wilson(k[best], n[best]); lo_w, hi_w = wilson(k[worst], n[worst])
    rows.append({'part': p, **{f'P{i}: failed/orders': f'{int(k[i])}/{int(n[i])}' for i in (0, 1, 2)}, 'best printer': best, 'worst printer': worst,
                 'best (%)': round(r[best] * 100, 1), 'worst (%)': round(r[worst] * 100, 1), 'intervals overlap': bool(hi_b >= lo_w)})
table6 = pd.DataFrame(rows).set_index('part')
display(table6)

by_part = orders.groupby('BOM Item')['failed'].agg(['sum', 'size']).reindex(part_order)
by_part['failed (%)'] = (by_part['sum'] / by_part['size'] * 100).round(1)
by_part['95 % interval (%)'] = [f'{lo * 100:.1f} - {hi * 100:.1f}' for lo, hi in (wilson(k, n) for k, n in zip(by_part['sum'], by_part['size']))]
print('failure rate per part, all printers together'); display(by_part.rename(columns={'sum': 'failed', 'size': 'orders'}))

fig, ax = plt.subplots(figsize=(9, 3.8))
rate6 = (tab['sum'] / tab['size'] * 100)
im = ax.imshow(rate6.values, cmap='Reds', aspect='auto', vmin=0)
ax.set_xticks(range(3)); ax.set_xticklabels([f'printer {i}' for i in range(3)]); ax.set_yticks(range(len(part_order))); ax.set_yticklabels(part_order)
for i in range(rate6.shape[0]):
    for j in range(3): ax.text(j, i, f'{rate6.iat[i, j]:.0f} %  (n = {int(tab[("size", j)].iloc[i])})', ha='center', va='center', fontsize=8, color='white' if rate6.iat[i, j] > 25 else 'black')
ax.set_title('Failed orders per part and printer'); plt.colorbar(im, label='% failed'); plt.tight_layout(); plt.show()
print(f'printer effect given the part (failed orders): permutation p = {p_fail:.2f}')

**Can you recommend a printer for any part? What would you recommend instead, and what data would change the answer?**

**No part can be assigned to a printer with a justification.** For every part the intervals of the best and the worst printer overlap, and the permutation test over all parts gives p = 0.20. The largest gaps are only apparent: part 2a on printer 2 (8 of 22 = 36 %) against printer 0 (4 of 20 = 20 %), part 3a on printer 2 (1 of 28) against printer 1 (8 of 41). With 5 to 40 orders per cell and a few failures, such gaps arise by chance.
**What the data do support instead** is a statement about the **parts**: failure rate over all printers 2a 25 % (95 % interval 16 to 38 %, 59 orders), 1a 16 % (13 to 19 %, 506 orders), 3a 14 %; 7a, 8a, 9a are very low (about 1 % or less). Since 1a is the largest share of the orders, it causes 80 of the 134 failed orders. The lever is therefore the **disturbance** behind failed orders (connection and extruder, question 4), not the allocation to printers.
**What would change the answer:** more orders per part and printer (especially 2a, 59 orders in total), the print settings and the maintenance state per order, and above all a **planned comparison**: allocating the same part alternately to two printers (randomised) would remove the confounding that the historical allocation always carries.

## Summary

**Summary of the six answers (number, uncertainty, limit of the data):**

| # | Answer | Uncertainty | What the data cannot say |
|---|---|---|---|
| 1 | Only **1a** shows more **warpage** (7.5 % vs 3.8 %) | other cells within chance | why (geometry, orientation) |
| 2 | **No** reliably different defect profile per printer | permutation p = 0.17; naive test misleading | printer effects of moderate size |
| 3 | **No** connection between blockage and interruption | p = 1.0 and 0.20; few events | causes (network, hot end) |
| 4 | 5,477 parts: 76.3 % OK, 10.4 % lost in failed orders, 11.0 % scrap, 2.2 % unknown | exact counts | why the disturbances occurred |
| 5 | **Disturbance pattern** differs (printer 0: connection, printer 1: blockages) | p = 6e-7; failed orders p = 0.20 | different part mix and operators |
| 6 | **No** printer assignment justified; parts 2a and 1a are the problem | all intervals overlap | needs a randomised comparison |

**General lesson:** diagnosis needs more than a table. Control for the part mix, respect that parts belong to orders, state the uncertainty, and accept “no evidence” as an answer. The unanswered *why* questions (red cards) need data that are not recorded today.